In [34]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

# Reproducibility
tf.keras.utils.set_random_seed(42)

# Configuration
DATASET_PATH = r"C:\Users\admin\Desktop\waste\final_merged_dataset"

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 26

os.makedirs("models", exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("Configuration loaded successfully!")


TensorFlow version: 2.21.0
Configuration loaded successfully!


In [35]:
class_names = sorted([
    folder.name
    for folder in Path(DATASET_PATH).iterdir()
    if folder.is_dir()
])

class_to_index = {
    name: index
    for index, name in enumerate(class_names)
}

image_extensions = {
    ".jpg", ".jpeg", ".png", ".bmp"
}

all_paths = []
all_labels = []

for class_name in class_names:

    folder_path = Path(DATASET_PATH) / class_name

    for image_path in folder_path.iterdir():

        if image_path.suffix.lower() in image_extensions:

            all_paths.append(str(image_path))
            all_labels.append(class_to_index[class_name])

print("Total Images:", len(all_paths))
print("Classes:", class_names)
print("Number of classes:", len(class_names))

Total Images: 19930
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Number of classes: 6


In [36]:
# 80% Training and 20% Temporary

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths,
    all_labels,
    test_size=0.20,
    random_state=SEED,
    stratify=all_labels
)

# Temporary 20% -> 10% Validation + 10% Testing

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_labels
)

print("Training Images:", len(train_paths))
print("Validation Images:", len(val_paths))
print("Testing Images:", len(test_paths))

Training Images: 15944
Validation Images: 1993
Testing Images: 1993


In [37]:
def check_distribution(labels):

    counts = Counter(labels)

    for index, name in enumerate(class_names):

        print(name, ":", counts[index])


print("TRAIN DISTRIBUTION")
check_distribution(train_labels)

print("\nVALIDATION DISTRIBUTION")
check_distribution(val_labels)

print("\nTEST DISTRIBUTION")
check_distribution(test_labels)

TRAIN DISTRIBUTION
cardboard : 2533
glass : 3742
metal : 2329
paper : 2797
plastic : 3118
trash : 1425

VALIDATION DISTRIBUTION
cardboard : 316
glass : 468
metal : 291
paper : 350
plastic : 389
trash : 179

TEST DISTRIBUTION
cardboard : 317
glass : 468
metal : 291
paper : 349
plastic : 390
trash : 178


In [38]:
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    image = tf.cast(image, tf.float32)

    image = image / 255.0

    return image, label


def create_dataset(paths, labels, training=False):

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:

        dataset = dataset.shuffle(
            buffer_size=len(paths),
            seed=SEED
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = create_dataset(
    train_paths,
    train_labels,
    training=True
)

val_ds = create_dataset(
    val_paths,
    val_labels
)

test_ds = create_dataset(
    test_paths,
    test_labels
)

print("All datasets created successfully!")

All datasets created successfully!


In [39]:
for images, labels in train_ds.take(1):

    print("Image shape:", images.shape)
    print("Label shape:", labels.shape)

    print(
        "Minimum pixel:",
        tf.reduce_min(images).numpy()
    )

    print(
        "Maximum pixel:",
        tf.reduce_max(images).numpy()
    )

Image shape: (32, 128, 128, 3)
Label shape: (32,)
Minimum pixel: 0.0
Maximum pixel: 1.0


In [40]:
classes = np.unique(train_labels)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels
)

class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(classes, weights)
}

print("Class Weights:")

for index, weight in class_weights.items():

    print(class_names[index], ":", weight)

Class Weights:
cardboard : 1.0490854059744703
glass : 0.710137181542847
metal : 1.1409760984685846
paper : 0.950065546418782
plastic : 0.852255719478298
trash : 1.864795321637427


In [41]:
model = models.Sequential([

    layers.Input(shape=(128, 128, 3)),

    # Block 1
    layers.Conv2D(
        32, (3, 3),
        activation="relu"
    ),

    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(
        64, (3, 3),
        activation="relu"
    ),

    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(
        128, (3, 3),
        activation="relu"
    ),

    layers.MaxPooling2D((2, 2)),

    # Classification
    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.3),

    layers.Dense(
        len(class_names),
        activation="softmax"
    )

])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,534 (431.77 KB)

 Trainable params: 110,534 (431.77 KB)

 Non-trainable params: 0 (0.00 B)

In [42]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

print("Model compiled successfully!")

Model compiled successfully!


In [43]:
callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        filepath="models/best_waste_cnn.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=0.00001,
        verbose=1
    )

]

In [44]:
history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=EPOCHS,

    class_weight=class_weights,

    callbacks=callbacks

)

Epoch 1/26


c:\Users\admin\Desktop\waste\waste_env\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 595ms/step - accuracy: 0.3096 - loss: 1.6258
Epoch 1: val_loss improved from None to 1.51746, saving model to models/best_waste_cnn.keras

Epoch 1: finished saving model to models/best_waste_cnn.keras
499/499 ━━━━━━━━━━━━━━━━━━━━ 321s 632ms/step - accuracy: 0.3096 - loss: 1.6258 - val_accuracy: 0.3798 - val_loss: 1.5175 - learning_rate: 0.0010
Epoch 2/26
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 585ms/step - accuracy: 0.4007 - loss: 1.4933
Epoch 2: val_loss improved from 1.51746 to 1.38387, saving model to models/best_waste_cnn.keras

Epoch 2: finished saving model to models/best_waste_cnn.keras
499/499 ━━━━━━━━━━━━━━━━━━━━ 303s 607ms/step - accuracy: 0.4007 - loss: 1.4933 - val_accuracy: 0.4561 - val_loss: 1.3839 - learning_rate: 0.0010
Epoch 3/26
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 549ms/step - accuracy: 0.4727 - loss: 1.3718
Epoch 3: val_loss improved from 1.38387 to 1.27793, saving model to models/best_waste_cnn.keras

Epoch 3: finished saving model to models/best

In [48]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test Loss:", test_loss)

print("Test Accuracy:", test_accuracy * 100, "%")

63/63 ━━━━━━━━━━━━━━━━━━━━ 11s 176ms/step - accuracy: 0.7737 - loss: 0.6518
Test Loss: 0.6518110036849976
Test Accuracy: 77.37079858779907 %


In [50]:
model.save("models/waste_classification_final_v1.keras")

import json

with open("models/class_names.json", "w") as file:

    json.dump(class_names, file)

print("Model and class names saved successfully!")

Model and class names saved successfully!


In [49]:

import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

MODEL_PATH = "models/waste_classification_final_v1.keras"

model = tf.keras.models.load_model(MODEL_PATH)

# Keep the same preprocessing used during training!
# val_ds should be your existing validation dataset.

loss, accuracy = model.evaluate(val_ds, verbose=1)

print(f"\nValidation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy * 100:.2f}%")

class_names = [
    "cardboard", "glass", "metal",
    "paper", "plastic", "trash"
]

y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))


63/63 ━━━━━━━━━━━━━━━━━━━━ 10s 149ms/step - accuracy: 0.7842 - loss: 0.6494

Validation Loss: 0.6494
Validation Accuracy: 78.42%

Classification Report:
              precision    recall  f1-score   support

   cardboard       0.83      0.88      0.85       316
       glass       0.85      0.81      0.83       468
       metal       0.74      0.82      0.78       291
       paper       0.77      0.84      0.80       350
     plastic       0.83      0.61      0.71       389
       trash       0.62      0.76      0.68       179

    accuracy                           0.78      1993
   macro avg       0.77      0.79      0.77      1993
weighted avg       0.79      0.78      0.78      1993


Confusion Matrix:
[[278   3   8  14  10   3]
 [ 13 377  20  19  24  15]
 [  6  10 239  23   3  10]
 [ 18   8  10 294   5  15]
 [ 14  40  33  22 239  41]
 [  7   4  15   9   8 136]]
